In [ ]:
# Core CV & tracking
!pip install -q ultralytics
!pip install -q lap

# Install a stable MediaPipe version
!pip uninstall -y mediapipe -q
!pip install -q mediapipe==0.10.14

# DeepFace dependencies
!pip install -q deepface
!pip install -q tf-keras

# ML
!pip install -q xgboost einops scikit-learn

# Visualization
!pip install -q seaborn matplotlib

print("✅ All dependencies installed")

In [ ]:
pip install "numpy<2.0" --force-reinstall

In [1]:
# import and global config
import os, sys, json, glob, re, math, warnings, random, time, logging
from pathlib import Path
from datetime import datetime, timedelta
from collections import defaultdict, deque
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from ultralytics import YOLO
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (roc_auc_score, classification_report,
                              confusion_matrix, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
logging.getLogger('deepface').setLevel(logging.ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print(f"Device      : {DEVICE}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: module 'numpy.dtypes' has no attribute 'StringDType'

In [ ]:
# path and constants
# ── Dataset paths ─────────────────────────────────────────────────────────────
# ── Dataset paths ─────────────────────────────────────────────────────────────
BASE_DIR   = Path('/kaggle/input/datasets/surajkumar1518/purchase-dataset/data')
VIDEO_DIR  = BASE_DIR / 'Videos_MERL_Shopping_Dataset'
LABEL_DIR  = BASE_DIR / 'Labels_MERL_Shopping_Dataset'
OUT_DIR    = Path('/kaggle/working/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── MERL action vocabulary ────────────────────────────────────────────────────
ACTION_MAP = {
    0: 'Background',
    1: 'Reach To Shelf',
    2: 'Retract From Shelf',
    3: 'Hand In Shelf',
    4: 'Inspect Product',   # ← strongest purchase signal
    5: 'Inspect Shelf',
}

# ── Video properties ──────────────────────────────────────────────────────────
FPS            = 30
FRAME_W        = 920
FRAME_H        = 680
SAMPLE_EVERY   = 3          # process 1 in every N frames (speed vs accuracy)
MAX_FRAMES     = 1800       # ~60 s per video; set None for full 2 min

# ── Window / sequence settings ────────────────────────────────────────────────
WINDOW_SEC     = 5          # seconds per feature window
WINDOW_FRAMES  = FPS * WINDOW_SEC   # 150 raw frames → 50 sampled
MAX_SEQ_LEN    = 24         # maximum windows per video (~2 min / 5 sec)

# ── Transformer hyper-parameters ──────────────────────────────────────────────
D_MODEL   = 128
N_HEADS   = 4
N_LAYERS  = 4
D_FF      = 256
DROPOUT   = 0.15
EPOCHS    = 40
BATCH     = 16
LR        = 3e-4

# ── Proximity thresholds (pixel units at 920×680) ─────────────────────────────
SOCIAL_DIST_PX  = 150       # max distance to consider two people interacting
SHELF_Y_RANGE   = (0.15, 0.75)  # normalised Y-band where the shelf is

print("✅ Configuration loaded")
print(f"   Window : {WINDOW_SEC}s  |  Max sequence : {MAX_SEQ_LEN} windows")
print(f"   D_MODEL: {D_MODEL}  |  Heads: {N_HEADS}  |  Layers: {N_LAYERS}")

In [ ]:
from pathlib import Path

base = Path("/kaggle/input/datasets/surajkumar1518/purchase-dataset/data")

for folder in base.iterdir():
    print(f"\n{folder.name}")
    files = list(folder.glob("*"))
    print("Number of files:", len(files))

    for f in files[:5]:
        print(" ", f.name)

In [ ]:
from pathlib import Path

label_dir = Path("/kaggle/input/datasets/surajkumar1518/purchase-dataset/data/Labels_MERL_Shopping_Dataset")

for f in list(label_dir.glob("*"))[:10]:
    print(f)

In [ ]:
from pathlib import Path

video_dir = Path("/kaggle/input/datasets/surajkumar1518/purchase-dataset/data/Videos_MERL_Shopping_Dataset")
test_dir  = Path("/kaggle/input/datasets/surajkumar1518/purchase-dataset/data/test")
valid_dir = Path("/kaggle/input/datasets/surajkumar1518/purchase-dataset/data/valid")
label_dir = Path("/kaggle/input/datasets/surajkumar1518/purchase-dataset/data/Labels_MERL_Shopping_Dataset")

n_videos = len(list(video_dir.glob("*")))
n_test   = len(list(test_dir.glob("*")))
n_valid  = len(list(valid_dir.glob("*")))

total_videos = n_videos + n_test + n_valid

print("Videos folder :", n_videos)
print("Test folder   :", n_test)
print("Valid folder  :", n_valid)
print("Total videos  :", total_videos)

print("Labels        :", len(list(label_dir.glob("*"))))

In [ ]:
from scipy.io import loadmat

mat = loadmat(
    "/kaggle/input/datasets/surajkumar1518/purchase-dataset/data/Labels_MERL_Shopping_Dataset/24_1_label.mat"
)

print(mat.keys())

In [ ]:
from scipy.io import loadmat
from pathlib import Path
import pandas as pd
import numpy as np

label_dir = Path("/kaggle/input/datasets/surajkumar1518/purchase-dataset/data/Labels_MERL_Shopping_Dataset")

rows = []

for mat_file in label_dir.glob("*.mat"):

    mat = loadmat(mat_file)

    label = mat["tlabs"]

    rows.append({
        "video_id": mat_file.stem.replace("_label", ""),
        "label_shape": str(label.shape),
        "label_value": label.flatten().tolist()
    })

df = pd.DataFrame(rows)

df.to_csv("/kaggle/working/all_labels.csv", index=False)

df.head()

In [ ]:
print(mat["tlabs"])
print(mat["tlabs"].shape)

In [ ]:
import os
print("Checking Kaggle Input Directory:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    if filenames:
        # Print the directory and the first file found inside it
        print(f"📁 {dirname}")
        print(f"   └─ Contains {len(filenames)} files (e.g., {filenames[0]})")

In [ ]:
import scipy.io as sio

# Grab the first .mat file in the folder
sample_file = list(LABEL_DIR.glob('*.mat'))[0]
mat_data = sio.loadmat(str(sample_file))

# Extract the actual data array
data_key = [k for k in mat_data.keys() if not k.startswith('__')][0]
raw_array = mat_data[data_key]

print(f"File Name   : {sample_file.name}")
print(f"Array Shape : {raw_array.shape}")
print("Raw Data (first 5 elements):")
print(raw_array[:5])

In [ ]:
import scipy.io as sio
import pandas as pd
from typing import Dict, Tuple
from pathlib import Path

# 1. The Fixed Parser (Handles the 5x1 array structure)
def parse_merl_mat_file(path: Path) -> pd.DataFrame:
    """Reads a MERL .mat file, extracting start/end frames grouped by Action ID."""
    mat_data = sio.loadmat(str(path))
    data_key = [k for k in mat_data.keys() if not k.startswith('__')][0]
    raw_array = mat_data[data_key]
    
    rows = []
    
    # The array shape is (5, 1). We loop through the 5 indices.
    for i in range(raw_array.shape[0]):
        action_id = i + 1  # Index 0 is Action 1, Index 1 is Action 2, etc.
        
        # Extract the 2D array of [start_frame, end_frame] pairs
        intervals = raw_array[i, 0] 
        
        # If the array isn't empty, loop through the pairs
        if len(intervals) > 0:
            for interval in intervals:
                sf, ef = int(interval[0]), int(interval[1])
                rows.append({
                    'action_id'  : action_id,
                    'action_name': ACTION_MAP.get(action_id, 'Unknown'),
                    'start_frame': sf,
                    'end_frame'  : ef,
                    'duration_f' : ef - sf,
                    'duration_s' : (ef - sf) / FPS,
                })
                
    df = pd.DataFrame(rows)
    
    # Safety net for videos with absolutely zero actions logged
    if df.empty:
        return pd.DataFrame(columns=[
            'action_id', 'action_name', 'start_frame', 
            'end_frame', 'duration_f', 'duration_s'
        ])
        
    df = df.sort_values('start_frame').reset_index(drop=True)
    return df


# 2. The Purchase Heuristic
def derive_purchase_label(label_df: pd.DataFrame) -> int:
    """Decides if the actions in the DataFrame result in a 'Buy' (1) or 'No Buy' (0)."""
    if label_df.empty:
        return 0
        
    has_inspect   = (label_df['action_id'] == 4).any()
    last_action   = label_df.sort_values('end_frame').iloc[-1]['action_id']
    inspect_dur   = label_df.loc[label_df['action_id']==4, 'duration_s'].sum()
    
    # The Buy Rule: Inspected item AND took it away AND inspected for >= 1 second
    if has_inspect and last_action == 2 and inspect_dur >= 2.0:
        return 1
    return 0


# 3. The Loop (This calculates and prints the ratio!)
def load_all_labels(label_dir: Path) -> Dict[str, Tuple[pd.DataFrame, int]]:
    records = {}
    for lf in sorted(label_dir.glob('*.mat')):
        df  = parse_merl_mat_file(lf)
        lbl = derive_purchase_label(df)
        records[lf.stem] = (df, lbl)
        
    # Calculate Buyer vs. Non-Buyer
    buy   = sum(v[1] for v in records.values())
    nobuy = len(records) - buy
    
    print(f"Loaded {len(records)} label files successfully.")
    if records:
        # Prints the ratio right here!
        print(f"  → BUY: {buy}  |  NO-BUY: {nobuy}  |  Ratio: {nobuy/max(buy,1):.1f}:1")
        
    return records


# 4. Execution & Preview
all_labels = load_all_labels(LABEL_DIR)

if all_labels:
    stem0  = next(iter(all_labels))
    df0, l0 = all_labels[stem0]
    print(f"\nPreview of first file — {stem0}  →  purchase={l0}")
    display(df0)
else:
    print("⚠️ WARNING: No labels were loaded.")

In [ ]:
import scipy.io as sio

# Grab the first .mat file in the folder
sample_file = list(LABEL_DIR.glob('*.mat'))[0]
mat_data = sio.loadmat(str(sample_file))

# Extract the actual data array
data_key = [k for k in mat_data.keys() if not k.startswith('__')][0]
raw_array = mat_data[data_key]

print(f"File Name   : {sample_file.name}")
print(f"Array Shape : {raw_array.shape}")
print("Raw Data (first 5 elements):")
print(raw_array[:5])

In [ ]:
# openCv frame utilities

def get_video_meta(path: Path) -> dict:
    cap = cv2.VideoCapture(str(path))
    meta = {
        'fps'    : cap.get(cv2.CAP_PROP_FPS),
        'frames' : int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        'width'  : int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height' : int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    }
    meta['duration_s'] = meta['frames'] / max(meta['fps'], 1)
    cap.release()
    return meta


def frame_generator(path: Path, max_frames: int = None,
                    every_n: int = 1, resize=None):
    """Yield (frame_idx, frame_bgr) from a video file."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise IOError(f"Cannot open: {path}")
    idx = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if idx % every_n == 0:
                if resize:
                    frame = cv2.resize(frame, resize)
                yield idx, frame
            idx += 1
            if max_frames and idx >= max_frames:
                break
    finally:
        cap.release()


def action_at_frame(label_df: pd.DataFrame, frame_idx: int) -> int:
    """Return MERL action ID active at frame_idx (0 = background)."""
    hit = label_df[(label_df['start_frame'] <= frame_idx) &
                   (label_df['end_frame']   >= frame_idx)]
    return int(hit.iloc[0]['action_id']) if not hit.empty else 0


# ── Verify dataset ─────────────────────────────────────────────────────────────
video_files = sorted(VIDEO_DIR.glob('*.avi'))
if not video_files:
    video_files = sorted(VIDEO_DIR.glob('*.mp4'))

print(f"Found {len(video_files)} video files")
if video_files:
    m = get_video_meta(video_files[0])
    print(f"Sample meta: {m}")


In [ ]:
# Create a list of the 60 video filenames (without the .mp4 extension)
available_videos = [v.stem.replace('_crop', '') for v in video_files] 
# Note: Since your videos are named '15_2_crop.mp4' and labels are '15_2_label.mat', 
# we might need to match just the '15_2' part.

# Filter the labels
matched_labels = {}
for stem, data in all_labels.items():
    # The label stem is '15_2_label', so we extract the '15_2' part
    base_name = stem.replace('_label', '') 
    
    # Check if a video with this base name exists in our 60 loaded videos
    if any(base_name in v.stem for v in video_files):
        matched_labels[base_name] = data

print(f"Matched {len(matched_labels)} labels to our 60 videos!")

## 6 · YOLOv8 Person Detector + ByteTrack Multi-Person Tracker

In [ ]:
import numpy as np
from typing import List, Optional
from ultralytics import YOLO

# Download YOLOv8 small — smarter model for overhead footage
yolo = YOLO('yolov8s.pt')

def detect_and_track(frame_bgr: np.ndarray, conf: float = 0.15) -> List[dict]:
    """
    Run YOLOv8 + ByteTrack on one frame.
    Filter removed to account for top-down CCTV misclassifications.
    """
    h, w = frame_bgr.shape[:2]
    
    # 1. Run the tracker WITHOUT the classes=[0] filter!
    results = yolo.track(
        source   = frame_bgr,
        conf     = conf,
        iou      = 0.5,
        tracker  = 'bytetrack.yaml',
        persist  = True,
        verbose  = False,
    )
    
    persons = []
    if results and results[0].boxes is not None:
        for box in results[0].boxes:
            if box.id is None:
                continue
                
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
            
            # 2. Size Filter: Ignore tiny objects (remotes, cats) that YOLO finds
            area = (x2-x1) * (y2-y1)
            if area < 5000:  
                continue
                
            tid  = int(box.id[0])
            conf_ = float(box.conf[0])
            cx, cy = (x1+x2)/2, (y1+y2)/2
            
            persons.append({
                'track_id' : tid,
                'bbox'     : [x1, y1, x2, y2],
                'conf'     : conf_,
                'cx' : cx, 'cy' : cy,
                'norm_cx'  : cx / w,
                'norm_cy'  : cy / h,
                'area'     : area,
            })
    return persons


def safe_crop(frame: np.ndarray, bbox: list,
              pad: float = 0.05) -> Optional[np.ndarray]:
    """Return person ROI with optional padding; None if invalid."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = bbox
    pw = (x2-x1) * pad
    ph = (y2-y1) * pad
    x1 = max(0, int(x1-pw));  y1 = max(0, int(y1-ph))
    x2 = min(w, int(x2+pw));  y2 = min(h, int(y2+ph))
    if x2 <= x1 or y2 <= y1:
        return None
    return frame[y1:y2, x1:x2]


print("✅ UPGRADED YOLO Tracker Ready")

## 7 · MediaPipe — Pose, Head Pose & Hand Analysis

In [ ]:
mp_pose  = mp.solutions.pose
mp_face  = mp.solutions.face_mesh
mp_hands = mp.solutions.hands

pose_est  = mp_pose.Pose(
    static_image_mode=False, model_complexity=1,
    min_detection_confidence=0.5, min_tracking_confidence=0.5)

face_est  = mp_face.FaceMesh(
    static_image_mode=False, max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5, min_tracking_confidence=0.5)

hand_est  = mp_hands.Hands(
    static_image_mode=False, max_num_hands=2,
    min_detection_confidence=0.5, min_tracking_confidence=0.5)

# ── Landmark shortcuts ────────────────────────────────────────────────────────
PL = mp_pose.PoseLandmark
NOSE        = PL.NOSE
L_SHOULDER  = PL.LEFT_SHOULDER;   R_SHOULDER = PL.RIGHT_SHOULDER
L_ELBOW     = PL.LEFT_ELBOW;      R_ELBOW    = PL.RIGHT_ELBOW
L_WRIST     = PL.LEFT_WRIST;      R_WRIST    = PL.RIGHT_WRIST
L_HIP       = PL.LEFT_HIP;        R_HIP      = PL.RIGHT_HIP
L_KNEE      = PL.LEFT_KNEE;       R_KNEE     = PL.RIGHT_KNEE
L_ANKLE     = PL.LEFT_ANKLE;      R_ANKLE    = PL.RIGHT_ANKLE


def _angle(a, b, c) -> float:
    """Joint angle at b formed by a-b-c (degrees)."""
    ba = np.array([a.x-b.x, a.y-b.y])
    bc = np.array([c.x-b.x, c.y-b.y])
    cos = np.dot(ba,bc) / (np.linalg.norm(ba)*np.linalg.norm(bc) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos,-1,1))))


def analyse_pose(crop: np.ndarray) -> dict:
    """
    Body language features from MediaPipe Pose.

    Features:
      pose_detected       – was a skeleton found?
      lean_angle          – torso lean in degrees (forward = larger)
      arm_cross           – 1 if wrists crossed (arms folded)
      step_back           – 1 if person leaning backward
      wrist_raised        – 1 if either wrist above shoulder (reaching)
      wrist_near_shelf    – 1 if wrist at shelf Y-band
      elbow_angle_l/r     – elbow flexion (degrees)
      shoulder_width_norm – shoulder width / image width (proxy for facing angle)
      body_turned_away    – 1 if shoulders nearly parallel to camera (side-on)
      relaxed_posture     – 1 if elbow angles > 120° (arms loose)
      holding_product     – 1 if wrist raised AND elbow bent < 100° (product held)
    """
    F = dict(pose_detected=0, lean_angle=0.0, arm_cross=0, step_back=0,
             wrist_raised=0, wrist_near_shelf=0,
             elbow_angle_l=0.0, elbow_angle_r=0.0,
             shoulder_width_norm=0.0, body_turned_away=0,
             relaxed_posture=0, holding_product=0)
    if crop is None or crop.size == 0:
        return F
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    res = pose_est.process(rgb)
    if not res.pose_landmarks:
        return F
    lm = res.pose_landmarks.landmark
    F['pose_detected'] = 1

    # Torso lean angle
    sh_mx = (lm[L_SHOULDER].x + lm[R_SHOULDER].x) / 2
    sh_my = (lm[L_SHOULDER].y + lm[R_SHOULDER].y) / 2
    hi_mx = (lm[L_HIP].x + lm[R_HIP].x) / 2
    hi_my = (lm[L_HIP].y + lm[R_HIP].y) / 2
    dx, dy = sh_mx - hi_mx, sh_my - hi_my
    F['lean_angle'] = float(np.degrees(np.arctan2(abs(dx), abs(dy)+1e-8)))
    F['step_back']  = int(sh_mx < hi_mx - 0.05)   # shoulder behind hip

    # Arm crossing
    F['arm_cross']  = int(lm[L_WRIST].x > lm[R_WRIST].x)

    # Wrist positions
    min_sh_y = min(lm[L_SHOULDER].y, lm[R_SHOULDER].y)
    lw_raised = lm[L_WRIST].y < min_sh_y
    rw_raised = lm[R_WRIST].y < min_sh_y
    F['wrist_raised'] = int(lw_raised or rw_raised)

    # Wrist near shelf Y-band (SHELF_Y_RANGE in normalised coords)
    for wl in [lm[L_WRIST], lm[R_WRIST]]:
        if SHELF_Y_RANGE[0] <= wl.y <= SHELF_Y_RANGE[1]:
            F['wrist_near_shelf'] = 1

    # Elbow angles
    F['elbow_angle_l'] = _angle(lm[L_SHOULDER], lm[L_ELBOW], lm[L_WRIST])
    F['elbow_angle_r'] = _angle(lm[R_SHOULDER], lm[R_ELBOW], lm[R_WRIST])

    # Shoulder width (camera-facing proxy)
    sw = abs(lm[L_SHOULDER].x - lm[R_SHOULDER].x)
    F['shoulder_width_norm'] = float(sw)
    F['body_turned_away']    = int(sw < 0.12)   # side-on to camera

    # Relaxed posture: both elbows loose
    F['relaxed_posture'] = int(F['elbow_angle_l'] > 120 and F['elbow_angle_r'] > 120)

    # Holding product: wrist raised + elbow bent
    bent = (F['elbow_angle_l'] < 100) or (F['elbow_angle_r'] < 100)
    F['holding_product'] = int(F['wrist_raised'] and bent)

    return F


def analyse_head_pose(crop: np.ndarray) -> dict:
    """
    Head orientation from FaceMesh landmarks.

    Features:
      face_detected       – FaceMesh found?
      head_yaw_proxy      – left/right rotation proxy (-1…+1)
      head_pitch_proxy    – up/down tilt proxy
      looking_at_product  – 1 if face roughly front-on (|yaw| < threshold)
      repeated_view_flag  – set externally by session tracker
    """
    F = dict(face_detected=0, head_yaw_proxy=0.0, head_pitch_proxy=0.0,
             looking_at_product=0)
    if crop is None or crop.size == 0:
        return F
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    res = face_est.process(rgb)
    if not res.multi_face_landmarks:
        return F
    F['face_detected'] = 1
    fl = res.multi_face_landmarks[0].landmark

    # Yaw: nose tip (1) relative to face centre (midpoint of ears 234/454)
    cx   = (fl[234].x + fl[454].x) / 2
    yaw  = fl[1].x - cx              # rightward positive
    F['head_yaw_proxy']      = float(np.clip(yaw / 0.15, -1, 1))

    # Pitch: nose tip relative to midpoint of forehead (10) & chin (152)
    cy    = (fl[10].y + fl[152].y) / 2
    pitch = fl[1].y - cy
    F['head_pitch_proxy']    = float(np.clip(pitch / 0.15, -1, 1))

    # Looking at product: roughly front-on and slightly down
    F['looking_at_product']  = int(abs(yaw) < 0.04 and -0.05 < pitch < 0.06)
    return F


def analyse_hands(crop: np.ndarray) -> dict:
    """
    Hand landmark features.

    Features:
      hands_detected     – number of hands found (0/1/2)
      hand_open          – 1 if fingers spread (product held loosely)
      hand_pinch         – 1 if thumb-index close (label reading)
    """
    F = dict(hands_detected=0, hand_open=0, hand_pinch=0)
    if crop is None or crop.size == 0:
        return F
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    res = hand_est.process(rgb)
    if not res.multi_hand_landmarks:
        return F
    F['hands_detected'] = len(res.multi_hand_landmarks)
    for hlm in res.multi_hand_landmarks:
        lm = hlm.landmark
        # Finger spread: distance index tip (8) to pinky tip (20)
        spread = math.hypot(lm[8].x-lm[20].x, lm[8].y-lm[20].y)
        if spread > 0.25:
            F['hand_open'] = 1
        # Pinch: thumb tip (4) close to index tip (8)
        pinch  = math.hypot(lm[4].x-lm[8].x, lm[4].y-lm[8].y)
        if pinch < 0.05:
            F['hand_pinch'] = 1
    return F


print("✅ MediaPipe analysers ready")


## 8 · DeepFace — Facial Expression Recognition

In [ ]:
from deepface import DeepFace

EMOTION_CLASSES = ['angry','disgust','fear','happy','sad','surprise','neutral']

# Map raw DeepFace emotions → project emotions
EMOTION_REMAP = {
    'happy'   : 'happy',
    'surprise': 'interested',
    'neutral' : 'neutral',
    'sad'     : 'disinterested',
    'angry'   : 'negative',
    'disgust' : 'negative',
    'fear'    : 'confused',
}

def analyse_emotion(crop: np.ndarray) -> dict:
    """
    Run DeepFace emotion analysis on person crop.
    Returns emotion scores as fractions summing to 1.
    Falls back to zeros if face not found.
    """
    base = {f'emo_{e}': 0.0 for e in
            ['happy','interested','neutral','disinterested','negative','confused']}
    base['emotion_detected'] = 0
    if crop is None or crop.size < 50*50*3:   # too small
        return base
    try:
        # enforce_detection=False avoids crash when face not clearly visible
        result = DeepFace.analyze(
            img_path       = crop,
            actions        = ['emotion'],
            enforce_detection = False,
            silent         = True,
        )
        if isinstance(result, list):
            result = result[0]
        raw_scores = result.get('emotion', {})
        total      = sum(raw_scores.values()) + 1e-8
        base['emotion_detected'] = 1
        for raw_em, score in raw_scores.items():
            mapped = EMOTION_REMAP.get(raw_em.lower(), 'neutral')
            base[f'emo_{mapped}'] = base.get(f'emo_{mapped}', 0.0) + score/total
    except Exception:
        pass   # face not found or model error — keep zeros
    return base


print("✅ DeepFace emotion analyser ready")
print("   Emotion classes:", list(EMOTION_REMAP.values()))


## 9 · Social Interaction Detector

In [ ]:
def compute_social_features(persons: List[dict],
                             frame_w: int, frame_h: int) -> Dict[int, dict]:
    """
    For each tracked person, compute social context features
    based on proximity to other people in the same frame.

    Returns dict: {track_id: {group_size, near_others, avg_group_dist}}
    """
    social = {}
    n = len(persons)
    for i, p in enumerate(persons):
        tid  = p['track_id']
        near = 0
        dists = []
        for j, q in enumerate(persons):
            if i == j:
                continue
            # Euclidean distance between bounding-box centres (pixels)
            d = math.hypot(p['cx']-q['cx'], p['cy']-q['cy'])
            if d < SOCIAL_DIST_PX:
                near  += 1
                dists.append(d)
        social[tid] = {
            'group_size'     : near + 1,       # self + companions
            'near_others'    : int(near > 0),
            'avg_group_dist' : float(np.mean(dists)) if dists else 0.0,
        }
    return social


print("✅ Social interaction detector ready")


## 10 · Customer Session — Behavioural Event Accumulator

In [ ]:
class CustomerSession:
    """
    Accumulates per-frame observations for one tracked customer.
    Every WINDOW_SEC seconds it emits one feature vector (a 'window').
    """

    def __init__(self, track_id: int):
        self.track_id    = track_id
        self._positions  = []   # (frame_idx, cx, cy)
        self._pose       = []
        self._head       = []
        self._hand       = []
        self._emotion    = []
        self._actions    = []   # MERL action IDs
        self._social     = []
        self.frame_count = 0
        self.window_idx  = 0
        self.first_frame = None
        self.last_frame  = None
        # Gaze state for 'repeated viewing'
        self._was_looking     = False
        self._look_switches   = 0

    def update(self, frame_idx, cx, cy, pose_f, head_f, hand_f,
               emotion_f, social_f, action_id):
        if self.first_frame is None:
            self.first_frame = frame_idx
        self.last_frame = frame_idx

        # Track repeated-view (look-away → look-back transitions)
        looking_now = bool(head_f.get('looking_at_product', 0))
        if looking_now != self._was_looking:
            self._look_switches += 1
        self._was_looking = looking_now

        self._positions.append((frame_idx, cx, cy))
        self._pose.append(pose_f)
        self._head.append(head_f)
        self._hand.append(hand_f)
        self._emotion.append(emotion_f)
        self._actions.append(action_id)
        self._social.append(social_f)
        self.frame_count += 1

    # ── helpers ───────────────────────────────────────────────────────────────
    @staticmethod
    def _mean(lst, key, default=0.0):
        vals = [d.get(key, default) for d in lst]
        return float(np.mean(vals)) if vals else default

    def _action_fracs(self) -> dict:
        total = max(len(self._actions), 1)
        cnts  = {a: 0 for a in range(6)}
        for a in self._actions:
            if a in cnts: cnts[a] += 1
        return {f'act_{k}_frac': v/total for k,v in cnts.items()}

    def _walk_stats(self) -> dict:
        pos = self._positions
        if len(pos) < 2:
            return dict(dwell_sec=len(pos)/FPS, mean_speed=0.0,
                        max_speed=0.0, direction_changes=0, stop_frac=0.0)
        speeds, dir_changes, stops = [], 0, 0
        prev_dir = None
        for i in range(1, len(pos)):
            dx = pos[i][1] - pos[i-1][1]
            dy = pos[i][2] - pos[i-1][2]
            sp = math.hypot(dx, dy)
            speeds.append(sp)
            d  = (np.sign(dx), np.sign(dy))
            if prev_dir and d != prev_dir:
                dir_changes += 1
            if sp < 1.5:
                stops += 1
            prev_dir = d
        return dict(
            dwell_sec        = len(pos) / FPS,
            mean_speed       = float(np.mean(speeds)),
            max_speed        = float(np.max(speeds)),
            direction_changes= dir_changes,
            stop_frac        = stops / max(len(speeds), 1),
        )

    # ── main feature builder ──────────────────────────────────────────────────
    def build_window_features(self) -> Optional[dict]:
        if not self._positions:
            return None
        F = {'track_id': self.track_id, 'window_idx': self.window_idx}

        # Walking & dwell
        F.update(self._walk_stats())

        # MERL action distribution
        F.update(self._action_fracs())

        # Body posture (MediaPipe Pose)
        for key in ['pose_detected','lean_angle','arm_cross','step_back',
                    'wrist_raised','wrist_near_shelf','elbow_angle_l',
                    'elbow_angle_r','shoulder_width_norm','body_turned_away',
                    'relaxed_posture','holding_product']:
            F[key] = self._mean(self._pose, key)

        # Head pose & gaze
        for key in ['face_detected','head_yaw_proxy','head_pitch_proxy',
                    'looking_at_product']:
            F[key] = self._mean(self._head, key)
        F['repeated_view_rate'] = self._look_switches / max(len(self._head), 1)

        # Hand
        for key in ['hands_detected','hand_open','hand_pinch']:
            F[key] = self._mean(self._hand, key)

        # Emotions
        for key in ['emotion_detected','emo_happy','emo_interested',
                    'emo_neutral','emo_disinterested','emo_negative','emo_confused']:
            F[key] = self._mean(self._emotion, key)

        # Social
        for key in ['group_size','near_others','avg_group_dist']:
            F[key] = self._mean(self._social, key)

        self.window_idx += 1
        return F

    def flush(self):
        """Emit current window and reset buffers."""
        feats = self.build_window_features()
        self._positions.clear(); self._pose.clear(); self._head.clear()
        self._hand.clear(); self._emotion.clear()
        self._actions.clear(); self._social.clear()
        return feats


print("✅ CustomerSession class ready")


## 11 · Full Video Processing Pipeline

In [ ]:
def process_video(video_path: Path,
                  label_df: pd.DataFrame,
                  purchase_label: int,
                  max_frames: int = MAX_FRAMES,
                  every_n: int = SAMPLE_EVERY,
                  enable_emotion: bool = False) -> List[dict]: # <-- FIX: Changed to False
    """
    Run the complete pipeline on one video.
    Returns a list of window-feature dicts (one per customer × 5-sec window).
    """
    sessions       : Dict[int, CustomerSession] = {}
    all_windows    : List[dict] = []
    frames_per_win = WINDOW_FRAMES // every_n   # sampled frames per window

    # Feed EVERY frame to YOLO (every_n=1) to keep ByteTrack stable
    for frame_idx, frame in frame_generator(video_path, max_frames, every_n=1):
        h, w = frame.shape[:2]

        # ── 1. Detect & track (Runs on EVERY frame) ──────────────────────────
        persons = detect_and_track(frame)
        if not persons:
            continue

        # ── 2. Downsample Heavy Processing ───────────────────────────────────
        # Skip the expensive MediaPipe/DeepFace steps if not a sampled frame
        if frame_idx % every_n != 0:
            continue

        # ── 3. Social context (all persons in frame) ─────────────────────────
        social_ctx = compute_social_features(persons, w, h)

        # ── 4. Per-person analysis ───────────────────────────────────────────
        action_id = action_at_frame(label_df, frame_idx)

        for person in persons:
            tid  = person['track_id']
            crop = safe_crop(frame, person['bbox'])

            if tid not in sessions:
                sessions[tid] = CustomerSession(tid)

            # MediaPipe
            pose_f    = analyse_pose(crop)
            head_f    = analyse_head_pose(crop)
            hand_f    = analyse_hands(crop)

            # Emotion (optional — expensive)
            if enable_emotion:
                emotion_f = analyse_emotion(crop)
            else:
                emotion_f = {f'emo_{e}': 0.0 for e in ['happy','interested','neutral','disinterested','negative','confused']}
                emotion_f['emotion_detected'] = 0

            # Retrieve social features for this specific person
            p_social = social_ctx.get(tid, {})

            # Call the update() method! 
            # This fills the CustomerSession buffers so flush() actually returns data.
            sessions[tid].update(
                frame_idx=frame_idx, 
                cx=person['cx'], 
                cy=person['cy'],
                pose_f=pose_f, 
                head_f=head_f, 
                hand_f=hand_f,
                emotion_f=emotion_f, 
                social_f=p_social, 
                action_id=action_id
            )

            # ── 5. Emit window every N sampled frames ────────────────────────
            # Add > 0 check so it doesn't accidentally trigger on frame 0
            if sessions[tid].frame_count > 0 and sessions[tid].frame_count % frames_per_win == 0:
                wf = sessions[tid].flush()
                if wf is not None:
                    wf['video_stem']     = video_path.stem
                    wf['purchase_label'] = purchase_label
                    all_windows.append(wf)

    # ── 6. Flush remaining partial windows at the end of the video ───────────
    for sess in sessions.values():
        if sess._positions:
            wf = sess.flush()
            if wf is not None:
                wf['video_stem']     = video_path.stem
                wf['purchase_label'] = purchase_label
                all_windows.append(wf)

    return all_windows

print("✅ process_video() corrected and ready")

## 12 · Run Pipeline on All 60 Videos

In [ ]:
sample_video = video_files[0]

cap = cv2.VideoCapture(str(sample_video))
ret, frame = cap.read()
cap.release()

plt.figure(figsize=(12,8))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
cap = cv2.VideoCapture(str(sample_video))

ret, frame = cap.read()

print(frame.shape)

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample_video = video_files[0]

cap = cv2.VideoCapture(str(sample_video))

for pos in [0, 200, 949, 1898, 2847]:

    cap.set(cv2.CAP_PROP_POS_FRAMES, pos)

    ret, frame = cap.read()

    if not ret:
        continue

    plt.figure(figsize=(10,8))
    plt.title(f"Frame {pos}")
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

cap.release()

In [ ]:
from ultralytics import YOLO
import cv2

# 1. Initialize the YOLO model (Downloads yolov8n.pt if you don't have it yet)
debug_yolo = YOLO('yolov8s.pt') # You can also use 'yolov8s.pt' for slightly better accuracy

cap = cv2.VideoCapture(str(sample_video))
cap.set(cv2.CAP_PROP_POS_FRAMES, 200)
ret, frame = cap.read()

if ret:
    # 2. Now debug_yolo exists and can process the frame
    results = debug_yolo(frame, conf=0.25)

    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])

        print(
            debug_yolo.names[cls_id],
            f"{conf:.2f}"
        )
else:
    print("Failed to read frame 200")

cap.release()

In [ ]:
cap = cv2.VideoCapture(str(sample_video))

cap.set(cv2.CAP_PROP_POS_FRAMES, 200)

ret, frame = cap.read()

results = debug_yolo(frame, conf=0.25)

for box in results[0].boxes:
    cls_id = int(box.cls[0])
    conf   = float(box.conf[0])

    print(
        debug_yolo.names[cls_id],
        f"{conf:.2f}"
    )

cap.release()

In [ ]:
test_frames = [150, 1898, 2847]

cap = cv2.VideoCapture(str(sample_video))

for fnum in test_frames:

    cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)

    ret, frame = cap.read()

    if not ret:
        continue

    results = debug_yolo(frame, conf=0.25, verbose=False)

    print(f"\nFrame {fnum}")

    if len(results[0].boxes) == 0:
        print("No detections")

    for box in results[0].boxes:

        cls = int(box.cls[0])
        conf = float(box.conf[0])

        print(
            f"{debug_yolo.names[cls]} "
            f"({conf:.2f})"
        )

cap.release()

In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(sample_video))

cap.set(cv2.CAP_PROP_POS_FRAMES, 200)

ret, frame = cap.read()

results = debug_yolo(frame, conf=0.25)

annotated = results[0].plot()

plt.figure(figsize=(12,8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

cap.release()

In [ ]:
cap = cv2.VideoCapture(str(sample_video))

person_frames = 0
frame_idx = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    if frame_idx % 30 == 0:  # check every 30th frame

        results = debug_yolo(frame, conf=0.25, verbose=False)

        found_person = False

        for box in results[0].boxes:

            cls = int(box.cls[0])

            if debug_yolo.names[cls] == "person":
                found_person = True
                break

        if found_person:
            person_frames += 1

    frame_idx += 1

cap.release()

print("Frames containing person:", person_frames)

In [ ]:
import cv2
from ultralytics import YOLO

debug_yolo = YOLO("yolov8s.pt")

cap = cv2.VideoCapture(str(sample_video))

start_frame = 200
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

for i in range(30):

    ret, frame = cap.read()

    if not ret:
        break

    results = debug_yolo(frame, conf=0.5, verbose=False)

    boxes = results[0].boxes

    if boxes is not None and len(boxes) > 0:

        for box in boxes:

            cls_id = int(box.cls[0])
            cls_name = debug_yolo.names[cls_id]
            conf = float(box.conf[0])

            print(
                f"Frame {start_frame+i}: "
                f"{cls_name} ({conf:.2f})"
            )

    else:
        print(f"Frame {start_frame+i}: No detections")

cap.release()

In [ ]:
import cv2
from ultralytics import YOLO

debug_yolo = YOLO("yolov8s.pt")

cap = cv2.VideoCapture(str(sample_video))

start_frame = 200
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

for i in range(30):

    ret, frame = cap.read()

    if not ret:
        break

    results = debug_yolo(frame, conf=0.5, verbose=False)

    boxes = results[0].boxes

    if boxes is not None and len(boxes) > 0:

        for box in boxes:

            cls_id = int(box.cls[0])
            cls_name = debug_yolo.names[cls_id]
            conf = float(box.conf[0])

            print(
                f"Frame {start_frame+i}: "
                f"{cls_name} ({conf:.2f})"
            )

    else:
        print(f"Frame {start_frame+i}: No detections")

cap.release()

In [ ]:
# Detect Only Persons

import cv2
from ultralytics import YOLO

debug_yolo = YOLO("yolov8s.pt")

cap = cv2.VideoCapture(str(sample_video))

frame_idx = 0
person_frames = []

while True:

    ret, frame = cap.read()

    if not ret:
        break

    if frame_idx % 10 == 0:  # every 10th frame

        results = debug_yolo(frame,
                             conf=0.5,
                             verbose=False)

        for box in results[0].boxes:

            cls_id = int(box.cls[0])

            if debug_yolo.names[cls_id] == "person":

                person_frames.append(frame_idx)
                break

    frame_idx += 1

cap.release()

print("First person frame:", min(person_frames))
print("Last person frame:", max(person_frames))
print("Total person frames:", len(person_frames))

In [ ]:
person_count = 0

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = debug_yolo(frame, conf=0.5, verbose=False)

    has_person = False

    for box in results[0].boxes:

        cls_id = int(box.cls[0])

        if debug_yolo.names[cls_id] == "person":
            has_person = True
            break

    if has_person:
        person_count += 1

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Process all videos with detailed debugging
# ─────────────────────────────────────────────────────────────────────────────

all_window_features = []
errors = []

print(f"Processing {len(video_files)} videos...")
t_start = time.time()

for vi, vp in enumerate(video_files):

    stem = match_label(vp.stem, all_labels)

    if stem is None:
        errors.append(f"{vp.name} (No matching label found)")
        print(f"[{vi+1:>3}/{len(video_files)}] SKIPPED: No label match")
        continue

    label_df, purchase = all_labels[stem]

    t0 = time.time()

    try:

        print(f"\n{'-'*60}")
        print(f"Video      : {vp.name}")
        print(f"Label File : {stem}")
        print(f"Purchase   : {purchase}")

        windows = process_video(vp, label_df, purchase)

        print(f"Windows Generated : {len(windows)}")

        if len(windows) == 0:
            print("⚠ WARNING: process_video returned zero windows")

        all_window_features.extend(windows)

        elapsed = time.time() - t0

        print(
            f"[{vi+1:>3}/{len(video_files)}] "
            f"{vp.name:<30} "
            f"buy={purchase} "
            f"wins={len(windows):>3} "
            f"{elapsed:.1f}s"
        )

    except Exception as e:

        errors.append(f"{vp.name}: {e}")

        print(
            f"[{vi+1:>3}/{len(video_files)}] "
            f"ERROR {vp.name}: {e}"
        )

total_time = time.time() - t_start

print("\n" + "="*60)

if len(all_window_features) == 0:

    print("⚠ CRITICAL: No windows were generated from any video")

    if errors:
        print("\nErrors:")
        for err in errors:
            print(" -", err)

else:

    feat_df = pd.DataFrame(all_window_features).fillna(0)

    feat_df.to_csv(
        OUT_DIR / "window_features.csv",
        index=False
    )

    print(f"Total windows : {len(feat_df)}")
    print(f"Total videos  : {feat_df['video_stem'].nunique()}")
    print(
        f"Buy / No-Buy  : "
        f"{feat_df['purchase_label'].sum()} / "
        f"{(feat_df['purchase_label']==0).sum()}"
    )

    print(f"Total time    : {total_time/60:.1f} min")

    if errors:
        print("\nErrors:")
        for err in errors:
            print(" -", err)

    display(feat_df.head())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Estimated runtime:
#   CPU only, every_n=3, max_frames=1800 per video:  ~12-18 min total
#   GPU (T4), same settings:                         ~5-7  min total
# ─────────────────────────────────────────────────────────────────────────────

# all_window_features : List[dict] = []
# errors : List[str] = []

# def match_label(stem: str, all_labels: dict):
#     """Clean match video stem (e.g. 15_2_crop) to label key (e.g. 15_2_label)."""
#     base_vid = stem.replace('_crop', '')
    
#     for k in all_labels:
#         base_lbl = k.replace('_label', '')
#         if base_vid == base_lbl:
#             return k
#     return None


# print(f"Processing {len(video_files)} videos...")
# t_start = time.time()

# for vi, vp in enumerate(video_files):
#     stem = match_label(vp.stem, all_labels)
    
#     # If no match is found, log the error and skip
#     if stem is None:
#         errors.append(f"{vp.name} (No matching label found)")
#         continue
        
#     label_df, purchase = all_labels[stem]

#     t0 = time.time()
#     try:
#         windows = process_video(vp, label_df, purchase)
#         all_window_features.extend(windows)
#         elapsed = time.time() - t0
#         print(f"  [{vi+1:>3}/{len(video_files)}] {vp.name:<30} "
#               f"buy={purchase}  wins={len(windows):>3}  {elapsed:.1f}s")
#     except Exception as e:
#         errors.append(f"{vp.name}: {e}")
#         print(f"  [{vi+1:>3}] ERROR {vp.name}: {e}")

# total_time = time.time() - t_start
# print(f"\n{'='*55}")

# # ── Safety Check before Pandas Processing ──
# if not all_window_features:
#     print("⚠️ CRITICAL: No windows were processed. feat_df is empty.")
#     if errors: 
#         print(f"Errors logged: {errors}")
# else:
#     feat_df = pd.DataFrame(all_window_features).fillna(0)
#     feat_df.to_csv(OUT_DIR / 'window_features.csv', index=False)
    
#     print(f"Total windows : {len(feat_df)}")
#     print(f"Total videos  : {feat_df['video_stem'].nunique()}")
#     print(f"Buy / No-Buy  : {feat_df['purchase_label'].sum():.0f} / "
#           f"{(feat_df['purchase_label']==0).sum()}")
#     print(f"Total time    : {total_time/60:.1f} min")
    
#     if errors: 
#         print(f"Errors        : {errors}")
        
#     display(feat_df.head(3))

In [ ]:
feat_df.to_parquet(OUT_DIR / "window_features.parquet", index=False)
feat_df.to_pickle(OUT_DIR / "window_features.pkl")
feat_df.to_csv(OUT_DIR / "window_features.csv", index=False)

print("All feature files saved successfully.")

In [ ]:
from IPython.display import FileLink, display

print("Click each link below to download:")
display(FileLink(r'output/window_features.parquet'))
display(FileLink(r'output/window_features.pkl'))
display(FileLink(r'output/window_features.csv'))

In [ ]:
import pandas as pd

# Load the highly-efficient parquet file
df = pd.read_parquet(OUT_DIR / "window_features.parquet")

print(f"Total 5-second windows : {len(df)}")
print(f"Total features per win : {len(df.columns)}")

print("\n--- Purchase Class Balance (Window-level) ---")
balance = df['purchase_label'].value_counts(normalize=True) * 100
print(balance.apply(lambda x: f"{x:.1f}%"))

print("\n--- Unique Videos Tracked ---")
print(f"Total Videos: {df['video_stem'].nunique()}")

# Peek at the column names to see our extracted features
print("\nSample Columns:")
print(list(df.columns)[:15])

## 13 · Exploratory Data Analysis

In [ ]:
# ── 13.1 Purchase distribution ────────────────────────────────────────────────
video_summary = (feat_df.groupby('video_stem')['purchase_label']
                         .first().value_counts())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(['No Buy (0)', 'Buy (1)'], video_summary.values,
            color=['#e74c3c','#2ecc71'], edgecolor='black', width=0.5)
axes[0].set_title('Purchase Distribution — per Video', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Videos')
for i, v in enumerate(video_summary.values):
    axes[0].text(i, v+0.3, str(v), ha='center', fontsize=12)

# ── 13.2 Action time distribution ─────────────────────────────────────────────
action_cols = [f'act_{i}_frac' for i in range(6) if f'act_{i}_frac' in feat_df.columns]
act_means   = feat_df.groupby('purchase_label')[action_cols].mean()
act_means.index = ['No Buy','Buy']
act_means.columns = [ACTION_MAP[int(c.split('_')[1])] for c in action_cols]
act_means.T.plot(kind='bar', ax=axes[1], color=['#e74c3c','#2ecc71'],
                 edgecolor='black')
axes[1].set_title('Mean Action Fraction: Buy vs No-Buy', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Fraction of Window')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_overview.png', dpi=130, bbox_inches='tight')
plt.show()


# ── 13.3 Feature distributions ────────────────────────────────────────────────
PLOT_FEATS = [
    ('dwell_sec',           'Dwell Time (s)'),
    ('wrist_raised',        'Wrist Raised Rate'),
    ('holding_product',     'Holding Product Rate'),
    ('looking_at_product',  'Looking at Product Rate'),
    ('arm_cross',           'Arm Cross Rate'),
    ('mean_speed',          'Mean Walking Speed (px/frame)'),
    ('emo_happy',           'Emotion: Happy'),
    ('relaxed_posture',     'Relaxed Posture Rate'),
    ('near_others',         'Near Others (Social)'),
]
PLOT_FEATS = [(c,t) for c,t in PLOT_FEATS if c in feat_df.columns]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle('Behavioural Feature Distributions: Buy vs No-Buy',
             fontsize=14, fontweight='bold', y=1.01)

for ax, (col, title) in zip(axes.flat, PLOT_FEATS):
    buy   = feat_df[feat_df['purchase_label']==1][col].dropna()
    nobuy = feat_df[feat_df['purchase_label']==0][col].dropna()
    ax.hist(nobuy, bins=25, alpha=0.55, color='#e74c3c', label='No Buy', density=True)
    ax.hist(buy,   bins=25, alpha=0.55, color='#2ecc71', label='Buy',    density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_distributions.png', dpi=130, bbox_inches='tight')
plt.show()


# ── 13.4 Correlation heatmap ──────────────────────────────────────────────────
num_cols = [c for c in feat_df.select_dtypes(include=np.number).columns
            if c not in ('track_id','window_idx')]
corr = feat_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.3, ax=ax, annot=False)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
from IPython.display import FileLink

# Create a download link for a specific image
display(FileLink(r'output/eda_overview.png'))
display(FileLink(r'output/feature_distributions.png'))

## 14 · Sequence Dataset for Transformer

In [ ]:
# Define feature columns (exclude metadata)
META_COLS  = {'video_stem', 'purchase_label', 'track_id', 'window_idx'}
FEAT_COLS  = [c for c in feat_df.select_dtypes(include=np.number).columns
              if c not in META_COLS]
N_FEATS    = len(FEAT_COLS)
print(f"Feature columns: {N_FEATS}")
print(FEAT_COLS)


class ShoppingDataset(Dataset):
    """
    Groups windows by (video_stem, track_id), sorts by window_idx.
    Pads / truncates to MAX_SEQ_LEN.
    Label is per video.
    """

    def __init__(self, df: pd.DataFrame, feat_cols: List[str],
                 max_seq: int, scaler: StandardScaler = None,
                 fit_scaler: bool = False):

        X_raw = df[feat_cols].fillna(0).values.astype(np.float32)

        if fit_scaler:
            self.scaler = StandardScaler().fit(X_raw)
        else:
            self.scaler = scaler
        X_sc = self.scaler.transform(X_raw)

        df2 = df.copy()
        df2[feat_cols] = X_sc

        self.items = []
        for (vstem, tid), grp in df2.groupby(['video_stem','track_id']):
            grp   = grp.sort_values('window_idx')
            label = int(grp['purchase_label'].iloc[0])
            seq   = grp[feat_cols].values.astype(np.float32)  # (T, F)
            T     = seq.shape[0]

            if T >= max_seq:
                seq  = seq[:max_seq]
                mask = np.ones(max_seq, bool)
            else:
                pad  = np.zeros((max_seq-T, len(feat_cols)), np.float32)
                seq  = np.vstack([seq, pad])
                mask = np.array([True]*T + [False]*(max_seq-T))

            self.items.append({
                'seq'  : torch.tensor(seq),              # (S, F)
                'mask' : torch.tensor(mask),             # (S,)
                'label': torch.tensor(label, dtype=torch.float32),
                'meta' : f"{vstem}_tid{tid}"
            })

    def __len__(self):        return len(self.items)
    def __getitem__(self, i): return self.items[i]


# ── Stratified train/val split on VIDEO level ─────────────────────────────────
video_label = feat_df.groupby('video_stem')['purchase_label'].first()
all_vstems  = video_label.index.tolist()
tr_vstems, va_vstems = train_test_split(
    all_vstems, test_size=0.2, random_state=42, stratify=video_label)

tr_df = feat_df[feat_df['video_stem'].isin(tr_vstems)]
va_df = feat_df[feat_df['video_stem'].isin(va_vstems)]

tr_ds = ShoppingDataset(tr_df, FEAT_COLS, MAX_SEQ_LEN, fit_scaler=True)
va_ds = ShoppingDataset(va_df, FEAT_COLS, MAX_SEQ_LEN,
                         scaler=tr_ds.scaler, fit_scaler=False)

tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  drop_last=True)
va_loader = DataLoader(va_ds, batch_size=BATCH, shuffle=False)

print(f"Train: {len(tr_ds)} sequences | Val: {len(va_ds)} sequences")
print(f"Train loader batches: {len(tr_loader)}")


## 15 · Transformer Encoder — Purchase Prediction Model

In [ ]:
class SinusoidalPE(nn.Module):
    """Sinusoidal positional encoding (fixed, not learned)."""
    def __init__(self, d_model: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model//2])
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, L, D)

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class BehaviourTransformer(nn.Module):
    """
    Transformer Encoder for customer behaviour sequence → purchase probability.

    Architecture:
      Input projection  : Linear(n_feats → d_model) + LayerNorm + GELU
      [CLS] token       : prepended learnable token
      Positional enc.   : sinusoidal
      Encoder layers    : N_LAYERS × (MultiHeadAttention + FFN + LayerNorm)
      CLS head          : Linear → GELU → Dropout → Linear → sigmoid
    """

    def __init__(self, n_feats: int,
                 d_model: int = D_MODEL, n_heads: int = N_HEADS,
                 n_layers: int = N_LAYERS, d_ff: int = D_FF,
                 max_seq: int = MAX_SEQ_LEN, dropout: float = DROPOUT):
        super().__init__()

        # Input projection
        self.proj = nn.Sequential(
            nn.Linear(n_feats, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # Positional encoding
        self.pos_enc = SinusoidalPE(d_model, max_seq + 1, dropout)

        # Transformer encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model        = d_model,
            nhead          = n_heads,
            dim_feedforward= d_ff,
            dropout        = dropout,
            activation     = 'gelu',
            batch_first    = True,
            norm_first     = True,   # Pre-LN for stability
        )
        self.encoder = nn.TransformerEncoder(
            enc_layer, num_layers=n_layers,
            norm=nn.LayerNorm(d_model))

        # Classification head
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor,
                mask: torch.Tensor = None) -> torch.Tensor:
        """
        x    : (B, T, F)  — behaviour feature sequence
        mask : (B, T)     — True = real token
        Returns: (B,)     — purchase logits
        """
        B, T, _ = x.shape

        # Project features
        x = self.proj(x)                                      # (B, T, D)

        # Prepend CLS
        cls = self.cls_token.expand(B, -1, -1)                # (B, 1, D)
        x   = torch.cat([cls, x], dim=1)                      # (B, T+1, D)
        x   = self.pos_enc(x)

        # Build key-padding mask for encoder (True = IGNORE)
        if mask is not None:
            cls_m   = torch.ones(B, 1, dtype=torch.bool, device=mask.device)
            full_m  = torch.cat([cls_m, mask], dim=1)         # (B, T+1)
            kp_mask = ~full_m                                  # flip: True=pad
        else:
            kp_mask = None

        # Encode
        out = self.encoder(x, src_key_padding_mask=kp_mask)   # (B, T+1, D)

        # CLS representation → logit
        cls_repr = out[:, 0]                                   # (B, D)
        logit    = self.head(cls_repr).squeeze(-1)             # (B,)
        return logit


# ── Instantiate ───────────────────────────────────────────────────────────────
model  = BehaviourTransformer(n_feats=N_FEATS).to(DEVICE)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters : {params:,}")
print(f"Input features   : {N_FEATS}")
print(f"d_model          : {D_MODEL}  |  Heads: {N_HEADS}  |  Layers: {N_LAYERS}")

# Sanity check
dummy = torch.randn(2, MAX_SEQ_LEN, N_FEATS).to(DEVICE)
dummy_mask = torch.ones(2, MAX_SEQ_LEN, dtype=torch.bool).to(DEVICE)
with torch.no_grad():
    out = model(dummy, dummy_mask)
print(f"Output shape     : {out.shape}  (expected: [2])")


## 16 · Training the Transformer

In [ ]:
# ── Class-imbalance weights ───────────────────────────────────────────────────
buy_n   = int(video_label.sum())
nobuy_n = len(video_label) - buy_n
pos_w   = torch.tensor([nobuy_n / max(buy_n, 1)], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

history = defaultdict(list)
best_auc, best_weights = 0.0, None

print(f"Training for {EPOCHS} epochs | device={DEVICE}")
print(f"pos_weight = {pos_w.item():.2f}  (no-buy : buy ratio)")
print("─"*55)

for epoch in range(1, EPOCHS+1):
    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    tr_losses, tr_preds, tr_labels = [], [], []
    for batch in tr_loader:
        seq   = batch['seq'].to(DEVICE)
        mask  = batch['mask'].to(DEVICE)
        label = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logit = model(seq, mask)
        loss  = criterion(logit, label)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        tr_losses.append(loss.item())
        tr_preds.extend(torch.sigmoid(logit).detach().cpu().tolist())
        tr_labels.extend(label.cpu().tolist())

    scheduler.step()

    # ── Validate ──────────────────────────────────────────────────────────────
    model.eval()
    va_losses, va_preds, va_labels = [], [], []
    with torch.no_grad():
        for batch in va_loader:
            seq   = batch['seq'].to(DEVICE)
            mask  = batch['mask'].to(DEVICE)
            label = batch['label'].to(DEVICE)
            logit = model(seq, mask)
            va_losses.append(criterion(logit, label).item())
            va_preds.extend(torch.sigmoid(logit).cpu().tolist())
            va_labels.extend(label.cpu().tolist())

    tr_loss = np.mean(tr_losses)
    va_loss = np.mean(va_losses)
    try:
        tr_auc = roc_auc_score(tr_labels, tr_preds)
        va_auc = roc_auc_score(va_labels, va_preds)
    except Exception:
        tr_auc = va_auc = 0.5

    history['tr_loss'].append(tr_loss)
    history['va_loss'].append(va_loss)
    history['tr_auc'].append(tr_auc)
    history['va_auc'].append(va_auc)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    if va_auc >= best_auc:
        best_auc     = va_auc
        best_weights = {k: v.clone() for k,v in model.state_dict().items()}

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}/{EPOCHS}  "
              f"tr_loss={tr_loss:.4f}  va_loss={va_loss:.4f}  "
              f"tr_AUC={tr_auc:.4f}  va_AUC={va_auc:.4f}  "
              f"lr={optimizer.param_groups[0]['lr']:.6f}")

# Restore best
model.load_state_dict(best_weights)
torch.save(best_weights, OUT_DIR / 'transformer_best.pt')
print(f"\n✅ Best Validation AUC: {best_auc:.4f}")


In [ ]:
# To prove the file is actually there
import os
from IPython.display import FileLink

file_path = OUT_DIR / 'transformer_best.pt'

# 1. Check if the file exists on the hard drive
if os.path.exists(file_path):
    print(f"✅ The file IS saved! Size: {os.path.getsize(file_path) / 1024:.1f} KB")
    
    # 2. Generate a clickable download link right here in the notebook
    display(FileLink(str(file_path)))
else:
    print("❌ The file is NOT there. Something went wrong in the training loop.")
    print(f"Files currently in OUT_DIR: {os.listdir(OUT_DIR)}")

In [ ]:
from IPython.display import FileLink, display

# Path to your saved model file
model_file_path = OUT_DIR / 'transformer_best.pt'

print("Click the link below to download your trained Transformer model:")
display(FileLink(str(model_file_path)))

In [ ]:
import os
import zipfile
from IPython.display import FileLink, display

model_source = '/kaggle/working/output/transformer_best.pt'
zip_destination = '/kaggle/working/transformer_best.zip'

if os.path.exists(model_source):
    # Pack the model file into a zip archive
    with zipfile.ZipFile(zip_destination, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(model_source, arcname='transformer_best.pt')
    
    print("✅ Model successfully packed into a zip archive!")
    print("Click the link below to download your weights directly:")
    
    # FIX: Pass the clean relative path since you're already in /kaggle/working
    display(FileLink('transformer_best.zip'))
else:
    print("❌ Could not find the file at '/kaggle/working/output/transformer_best.pt'")

## 17 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_x = range(1, len(history['tr_loss'])+1)

# Loss
axes[0].plot(epochs_x, history['tr_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_x, history['va_loss'], label='Val',   color='tomato')
axes[0].set_title('BCE Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend()

# AUC
axes[1].plot(epochs_x, history['tr_auc'], label='Train AUC', color='steelblue')
axes[1].plot(epochs_x, history['va_auc'], label='Val AUC',   color='tomato')
axes[1].axhline(best_auc, color='green', ls='--', lw=1.5,
                label=f'Best={best_auc:.3f}')
axes[1].set_title('ROC-AUC', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend()

# LR
axes[2].plot(epochs_x, history['lr'], color='purple')
axes[2].set_title('Learning Rate Schedule', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=130, bbox_inches='tight')
plt.show()


## 18 · Model Evaluation

In [ ]:
model.eval()
all_probs, all_true = [], []
with torch.no_grad():
    for batch in va_loader:
        logit = model(batch['seq'].to(DEVICE), batch['mask'].to(DEVICE))
        all_probs.extend(torch.sigmoid(logit).cpu().tolist())
        all_true.extend(batch['label'].tolist())

all_probs = np.array(all_probs)
all_true  = np.array(all_true)
all_preds = (all_probs >= 0.5).astype(int)

# ── Threshold sensitivity (Youden's J) ───────────────────────────────────────
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(all_true, all_probs)
j_scores = tpr - fpr
best_thresh = float(thresholds[np.argmax(j_scores)])
opt_preds   = (all_probs >= best_thresh).astype(int)

auc_score = roc_auc_score(all_true, all_probs)
print(f"Transformer Validation AUC  : {auc_score:.4f}")
print(f"Optimal threshold (Youden J): {best_thresh:.3f}")
print()
print("── Default threshold (0.50) ──────────────────────")
print(classification_report(all_true, all_preds, target_names=['No Buy','Buy']))
print("── Optimal threshold ─────────────────────────────")
print(classification_report(all_true, opt_preds, target_names=['No Buy','Buy']))


# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC curve
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC={auc_score:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].scatter(fpr[np.argmax(j_scores)], tpr[np.argmax(j_scores)],
                color='red', zorder=5, label=f'Best t={best_thresh:.2f}')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve', fontweight='bold'); axes[0].legend()

# Confusion matrix (optimal threshold)
cm = confusion_matrix(all_true, opt_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No Buy','Buy'], yticklabels=['No Buy','Buy'])
axes[1].set_title(f'Confusion Matrix (t={best_thresh:.2f})', fontweight='bold')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

# Probability histogram
axes[2].hist(all_probs[all_true==0], bins=20, alpha=0.6,
             color='#e74c3c', label='No Buy', density=True)
axes[2].hist(all_probs[all_true==1], bins=20, alpha=0.6,
             color='#2ecc71', label='Buy',    density=True)
axes[2].axvline(best_thresh, color='black', ls='--', label=f't={best_thresh:.2f}')
axes[2].set_title('Predicted Probability Distribution', fontweight='bold')
axes[2].set_xlabel('P(Buy)'); axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'evaluation.png', dpi=130, bbox_inches='tight')
plt.show()


## 19 · XGBoost Baseline — Aggregated Features

In [ ]:
# Per-video aggregation: mean, max, std of all window features
agg_ops = {}
for col in FEAT_COLS:
    for fn in ['mean','max','std']:
        agg_ops[f'{col}__{fn}'] = pd.NamedAgg(column=col, aggfunc=fn)

video_agg = (feat_df.groupby('video_stem')
                     .agg(**agg_ops)
                     .reset_index()
                     .fillna(0))
video_agg = video_agg.merge(
    feat_df[['video_stem','purchase_label']].drop_duplicates(), on='video_stem')

XGB_FEATS = [c for c in video_agg.columns
             if c not in ('video_stem','purchase_label')]
X = video_agg[XGB_FEATS].values
y = video_agg['purchase_label'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

# --- THE FIX: Calculate counts dynamically from the training labels ---
buy_count = (y_tr == 1).sum()
nobuy_count = (y_tr == 0).sum()
# --------------------------------------------------------------------

xgb_clf = xgb.XGBClassifier(
    n_estimators     = 400,
    max_depth        = 4,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 3,
    gamma            = 0.1,
    reg_lambda       = 1.5,
    scale_pos_weight = nobuy_count / max(buy_count, 1), # Now these variables exist!
    eval_metric      = 'auc',
    use_label_encoder= False,
    random_state     = 42,
    device           = 'cuda' if torch.cuda.is_available() else 'cpu',
)

xgb_clf.fit(X_tr, y_tr,
            eval_set=[(X_te, y_te)],
            verbose=50)

xgb_probs = xgb_clf.predict_proba(X_te)[:,1]
xgb_preds = (xgb_probs >= 0.5).astype(int)
xgb_auc   = roc_auc_score(y_te, xgb_probs)

print(f"\nXGBoost Validation AUC : {xgb_auc:.4f}")
print(classification_report(y_te, xgb_preds, target_names=['No Buy','Buy']))

## 21 · Ensemble: Transformer + XGBoost

In [ ]:
# Build XGB features for val videos
va_agg = (va_df.groupby('video_stem')
               .agg(**agg_ops)
               .reset_index()
               .fillna(0))
va_agg = va_agg.merge(
    va_df[['video_stem','purchase_label']].drop_duplicates(), on='video_stem')

X_va  = va_agg[XGB_FEATS].values
y_va  = va_agg['purchase_label'].values
xgb_va_probs = xgb_clf.predict_proba(X_va)[:,1]

# Align transformer probs to same video order
va_ds_probs = {}
model.eval()
with torch.no_grad():
    for batch in va_loader:
        logits = model(batch['seq'].to(DEVICE), batch['mask'].to(DEVICE))
        probs  = torch.sigmoid(logits).cpu().tolist()
        for meta, prob in zip(batch['meta'], probs):
            vstem = meta.rsplit('_tid',1)[0]
            va_ds_probs.setdefault(vstem, []).append(prob)

# Average transformer prob per video
tf_va_probs = np.array([np.mean(va_ds_probs.get(vs, [0.5]))
                         for vs in va_agg['video_stem']])

for alpha in [0.3, 0.5, 0.7]:
    ens = alpha * tf_va_probs + (1-alpha) * xgb_va_probs
    auc = roc_auc_score(y_va, ens)
    print(f"Ensemble α={alpha:.1f} (TF) + {1-alpha:.1f} (XGB)  →  AUC={auc:.4f}")


## 22 · Real-Time Inference on a New Video

In [ ]:
import cv2
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional
from collections import defaultdict

def infer_video(video_path: Path,
                model: nn.Module,
                scaler: StandardScaler,
                feat_cols: List[str],
                threshold: float = 0.5,
                max_frames: int = MAX_FRAMES,
                every_n: int = SAMPLE_EVERY,
                annotate: bool = True,
                out_path: Optional[Path] = None) -> Dict[int, float]:
    """
    End-to-end inference on a single video.
    Returns {track_id: purchase_probability}.
    If annotate=True, saves annotated MP4 to out_path with live probability updates.
    """
    sessions: Dict[int, CustomerSession] = {}
    windows_by_track: Dict[int, List[dict]] = defaultdict(list)
    frames_per_win = WINDOW_FRAMES // every_n
    blank_label_df = pd.DataFrame(columns=['action_id','start_frame','end_frame'])

    # ── Optional video writer ─────────────────────────────────────────────────
    writer = None
    if annotate and out_path:
        meta   = get_video_meta(video_path)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(str(out_path), fourcc, 15,
                                 (meta['width'], meta['height']))

    predictions: Dict[int, float] = {}   # Updated live as windows complete

    for frame_idx, frame in frame_generator(video_path, max_frames, every_n):
        persons    = detect_and_track(frame)
        social_ctx = compute_social_features(persons, *frame.shape[1::-1])

        for p in persons:
            tid  = p['track_id']
            crop = safe_crop(frame, p['bbox'])
            if tid not in sessions:
                sessions[tid] = CustomerSession(tid)

            # ── FIX 1: Corrected dictionary comprehension syntax ──────────────
            emotion_default = {
                **{f'emo_{e}': 0.0 for e in ['happy','interested','neutral','disinterested','negative','confused']},
                'emotion_detected': 0
            }

            sessions[tid].update(
                frame_idx, p['cx'], p['cy'],
                analyse_pose(crop),
                analyse_head_pose(crop),
                analyse_hands(crop),
                emotion_default,
                social_ctx.get(tid, {'group_size':1,'near_others':0,'avg_group_dist':0.0}),
                action_id=0
            )

            # ── FIX 2: Live Inference when a 5-second window finishes ─────────
            if sessions[tid].frame_count % frames_per_win == 0:
                wf = sessions[tid].flush()
                if wf: 
                    windows_by_track[tid].append(wf)
                    
                    # Generate real-time prediction for the video annotation writer
                    df_tmp = pd.DataFrame(windows_by_track[tid]).fillna(0)
                    df_tmp['video_stem'] = 'inference'
                    df_tmp['purchase_label'] = 0
                    df_tmp['track_id'] = tid
                    
                    for c in feat_cols:
                        if c not in df_tmp.columns: df_tmp[c] = 0.0
                        
                    ds_tmp = ShoppingDataset(df_tmp, feat_cols, MAX_SEQ_LEN, scaler=scaler, fit_scaler=False)
                    if len(ds_tmp) > 0:
                        item = ds_tmp[0]
                        seq  = item['seq'].unsqueeze(0).to(DEVICE)
                        msk  = item['mask'].unsqueeze(0).to(DEVICE)
                        model.eval()
                        with torch.no_grad():
                            prob = torch.sigmoid(model(seq, msk)).item()
                        predictions[tid] = round(prob, 4)

        # ── Annotate frame (Now pulls live probabilities!) ────────────────────
        if writer:
            for p in persons:
                tid  = p['track_id']
                prob = predictions.get(tid, None)
                x1, y1, x2, y2 = [int(v) for v in p['bbox']]
                colour = (0,200,0) if prob and prob >= threshold else (0,80,200)
                cv2.rectangle(frame, (x1,y1), (x2,y2), colour, 2)
                txt = f"ID:{tid}"
                if prob is not None:
                    txt += f" P(buy):{prob:.0%}"
                cv2.putText(frame, txt, (x1, max(y1-8,10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2)
            writer.write(frame)

    # Flush remaining trailing frames
    for tid, sess in sessions.items():
        if sess._positions:
            wf = sess.flush()
            if wf: windows_by_track[tid].append(wf)

    if writer: writer.release()

    # Final terminal calculation pass for partial/remaining timelines
    model.eval()
    for tid, wlist in windows_by_track.items():
        if not wlist: continue
        df_tmp = pd.DataFrame(wlist).fillna(0)
        df_tmp['video_stem'] = 'inference'
        df_tmp['purchase_label'] = 0
        df_tmp['track_id'] = tid

        for c in feat_cols:
            if c not in df_tmp.columns: df_tmp[c] = 0.0

        ds_tmp = ShoppingDataset(df_tmp, feat_cols, MAX_SEQ_LEN, scaler=scaler, fit_scaler=False)
        if len(ds_tmp) == 0: continue
        item = ds_tmp[0]
        seq  = item['seq'].unsqueeze(0).to(DEVICE)
        msk  = item['mask'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prob = torch.sigmoid(model(seq, msk)).item()
        predictions[tid] = round(prob, 4)

    return predictions

In [ ]:
import os
import time

# Verify we have videos to test on
if 'video_files' in locals() and len(video_files) > 0:
    
    # 1. Pick the first video as our test subject
    demo_video = video_files[1]
    demo_out   = OUT_DIR / 'annotated_demo.mp4'
    
    print(f"🚀 Starting Real-Time Inference on: {demo_video.name}")
    print("⏳ Processing frames (YOLO + MediaPipe + Transformer)... Please wait.")
    
    t_start = time.time()
    
    # 2. Call the function and catch the returned predictions
    # We use best_thresh (0.725) that you calculated earlier!
    preds = infer_video(
        video_path=demo_video, 
        model=model, 
        scaler=tr_ds.scaler, 
        feat_cols=FEAT_COLS,
        threshold=best_thresh,
        annotate=True, 
        out_path=demo_out
    )
    
    elapsed_time = time.time() - t_start
    print(f"📊 Inference completed in {elapsed_time:.1f} seconds!")
    print("=" * 50)
    
    # 3. Print out the results beautifully
    print(f"\n🛍️ Purchase Predictions for {demo_video.name}:")
    print(f"{'Customer ID':<14} {'P(Buy)':<12} {'AI Verdict'}")
    print("─" * 45)
    
    if not preds:
        print("  🚶 No shoppers were tracked long enough to generate predictions.")
    else:
        for tid, prob in sorted(preds.items()):
            decision = "✅ LIKELY BUY" if prob >= best_thresh else "❌ NO BUY"
            print(f"  ID {tid:<10} {prob * 100:.2f}%          {decision}")
            
    # 4. Confirm the video file was saved completely
    if demo_out.exists():
        file_size = os.path.getsize(demo_out) / (1024 * 1024)
        print(f"\n🎬 Annotated MP4 saved successfully!")
        print(f"   📁 Path: {demo_out}")
        print(f"   📦 Size: {file_size:.2f} MB")
        
else:
    print("❌ Error: 'video_files' list is empty or not loaded in memory. Run your setup cells first!")

## 23 · Attention Weight Visualisation

In [ ]:
def get_attention_weights(model: nn.Module,
                           seq: torch.Tensor,
                           mask: torch.Tensor) -> np.ndarray:
    """
    Extract averaged attention weights from all layers.
    Returns (n_heads * n_layers, T+1, T+1) averaged attention.
    Uses PyTorch forward hooks.
    """
    attn_maps = []

    def hook_fn(module, inp, out):
        # TransformerEncoderLayer returns output only; we need attn weights
        # Re-run attention manually for visualisation
        pass   # Hooks on standard nn.TransformerEncoderLayer don't expose attn

    # Simpler approach: use the built-in need_weights path
    B, T, F = seq.shape
    x   = model.proj(seq)
    cls = model.cls_token.expand(B, -1, -1)
    x   = torch.cat([cls, x], dim=1)
    x   = model.pos_enc(x)

    if mask is not None:
        cls_m = torch.ones(B,1,dtype=torch.bool,device=mask.device)
        kpm   = ~torch.cat([cls_m, mask], dim=1)
    else:
        kpm   = None

    attn_w_all = []
    for layer in model.encoder.layers:
        # Access self-attention sub-module directly
        sa = layer.self_attn
        _, attn_w = sa(x, x, x, key_padding_mask=kpm, need_weights=True,
                       average_attn_weights=True)
        attn_w_all.append(attn_w.detach().cpu().numpy())  # (B, T+1, T+1)
        # Pass through the layer normally to update x
        with torch.no_grad():
            x = layer(x, src_key_padding_mask=kpm)

    return np.array(attn_w_all)   # (n_layers, B, T+1, T+1)


# ── Visualise on first val sample ─────────────────────────────────────────────
model.eval()
sample_batch = next(iter(va_loader))
seq_s  = sample_batch['seq'][:1].to(DEVICE)
mask_s = sample_batch['mask'][:1].to(DEVICE)
label_s= int(sample_batch['label'][0])

with torch.no_grad():
    attn = get_attention_weights(model, seq_s, mask_s)   # (L, 1, T+1, T+1)

fig, axes = plt.subplots(1, N_LAYERS, figsize=(5*N_LAYERS, 4))
for li, ax in enumerate(axes):
    im = ax.imshow(attn[li, 0], aspect='auto', cmap='hot_r', vmin=0)
    ax.set_title(f'Layer {li+1}', fontweight='bold')
    ax.set_xlabel('Key (window idx)')
    ax.set_ylabel('Query (window idx)')
    plt.colorbar(im, ax=ax, fraction=0.04)

fig.suptitle(f'Self-Attention Weights  |  True label: {"BUY" if label_s else "NO BUY"}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'attention_weights.png', dpi=130, bbox_inches='tight')
plt.show()


In [ ]:
import math
import torch
import torch.nn as nn
from pathlib import Path

# ── 1. FIXED BLUEPRINT ARCHITECTURE (Matches your exact checkpoint) ──────────

class SinusoidalPE(nn.Module):
    def __init__(self, d_model: int, max_len: int, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:d_model//2])
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.drop(x + self.pe[:, :x.size(1)])


class BehaviourTransformer(nn.Module):
    def __init__(self, n_feats: int,
                 d_model: int = 128, n_heads: int = 4,
                 n_layers: int = 4, d_ff: int = 256,
                 max_seq: int = 24, dropout: float = 0.1): # <-- FIX: Max seq set to 24
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(n_feats, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_enc = SinusoidalPE(d_model, max_seq + 1, dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers, norm=nn.LayerNorm(d_model))

        # <-- FIX: Removed the extra LayerNorm to match your unexpected/missing key signature
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        B, T, _ = x.shape
        x = self.proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.pos_enc(x)

        if mask is not None:
            cls_m = torch.ones(B, 1, dtype=torch.bool, device=mask.device)
            full_m = torch.cat([cls_m, mask], dim=1)
            kp_mask = ~full_m
        else:
            kp_mask = None

        out = self.encoder(x, src_key_padding_mask=kp_mask)
        return self.head(out[:, 0]).squeeze(-1)


# ── 2. INSTANTIATE AND LOAD WEIGHTS UNCOMPROMISED ───────────────────────────

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = Path('/kaggle/input/models/surajkumar1518/purchase/transformers/default/1/transformer_best.pt')

print(f"🧠 Rebuilding layout topology (n_feats=41, max_seq=24)...")
# Force the exact structural requirements found in the checkpoint matrix
model = BehaviourTransformer(n_feats=41, max_seq=24).to(DEVICE)

print(f"📥 Loading weights from Registry Input path...")
loaded_weights = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)

# Pour weights into the newly rebuilt, exact structural match skeleton
model.load_state_dict(loaded_weights)
model.eval()

print(f"\n✅ SUCCESS: Model architecture completely aligned and ready on: {DEVICE}!")

In [ ]:
import os
import joblib
from pathlib import Path
from IPython.display import FileLink, display

# 1. Establish a clear output directory
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Pull weights to CPU memory space for maximum deployment portability
# This keeps the file safe to load on any device layout later
portable_state_dict = {k: v.cpu() for k, v in model.state_dict().items()}

# 3. Define the destination file path
joblib_file_path = OUTPUT_DIR / 'transformer_behaviour_best.joblib'

# 4. Dump and compress the state dictionary matrices
print("💾 Serializing and compressing model weights to Joblib format...")
joblib.dump(portable_state_dict, joblib_file_path, compress=3) # Level 3 compression balances speed and size

# 5. Output confirmation and performance metrics
file_size_mb = os.path.getsize(joblib_file_path) / (1024 * 1024)
print(f"✅ Success! Model weights successfully serialized.")
print(f"   📦 Saved Path : {joblib_file_path}")
print(f"   📊 File Size  : {file_size_mb:.2f} MB")

# 6. Generate direct download link
print("\n⬇️ Click the link below to download your new Joblib file directly:")
display(FileLink('output/transformer_behaviour_best.joblib'))